In [ ]:
DATASET = r"Z:\Joseph\250528_B2_003"   # <-- point at your dataset folder

import sys
sys.path.insert(0, r"Z:\Joseph\orgpipe")
from pathlib import Path
from orgpipe import config, layout
ds = Path(DATASET)
cfg = config.load_config(ds)
print("frame rate:", config.resolve_frame_rate(ds, cfg))

In [ ]:
# --- run CNMF-E (identical to the batch stage; SMOKE=True for a 300-frame test) ---
SMOKE = False
import os
temp = layout.caiman_temp(ds); temp.mkdir(parents=True, exist_ok=True)
os.environ["CAIMAN_TEMP"] = str(temp)
from orgpipe import stage_denoise
cnm = stage_denoise.fit(ds, cfg, smoke=SMOKE)

In [ ]:
# --- inspect components (the old notebook's cell 3, interactive-only) ---
import matplotlib.pyplot as plt
import caiman as cm
n_preview = 300 if SMOKE else 1000
movie = cm.load(str(layout.orgpipe_dir(ds) / "smoke_input.tif") if SMOKE
                else str(layout.raw_tif(ds)), subindices=range(0, n_preview))
corr_img = movie.local_correlations(swap_dim=False)
if cnm.estimates.idx_components is not None and len(cnm.estimates.idx_components):
    cnm.estimates.plot_contours(img=corr_img, idx=cnm.estimates.idx_components)
plt.show()
cnm.estimates.view_components(img=corr_img)

In [ ]:
# --- save the denoised movie (chunked float32) ---
stage_denoise.write_denoised(cnm, layout.denoised_tif(ds), chunk=cfg["denoise"]["chunk_size"])
print("saved", layout.denoised_tif(ds))